In [ ]:
from collections import defaultdict
import pickle
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn.functional as F
import torch.nn as nn

from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'



In [ ]:
#класс датасета для модели
class GraphDataset(Dataset):
    def __init__(self, h_local, h_type, rel, t_local, t_type):
        self.h_local = h_local
        self.h_type = h_type
        self.rel = rel
        self.t_local = t_local
        self.t_type = t_type

    def __len__(self):
        return len(self.h_local)

    def __getitem__(self, idx):
        return (
            self.h_local[idx],
            self.h_type[idx],
            self.rel[idx],
            self.t_local[idx],
            self.t_type[idx]
        )

#класс модели
class MultiModalDistMult(nn.Module):
    def __init__(self, raw_tensors, output_dim, num_relations):
        super().__init__()
        self.output_dim = output_dim
        self.num_types = len(raw_tensors)

        #хранилище для сырых эмбеддингов
        self.raw_stores = nn.ModuleDict({
            str(t_id): nn.Embedding.from_pretrained(tensor, freeze=True)
            for t_id, tensor in raw_tensors.items()
        })

        #создаем MLP-проекторы автоматически под размер входа каждого тензора
        self.projections = nn.ModuleDict()
        for t_id, tensor in raw_tensors.items():
            in_dim = tensor.shape[1] #узнаем исходную размерность одного типа
            self.projections[str(t_id)] = nn.Sequential( #создаем проектор
                nn.Linear(in_dim, output_dim * 2),
                nn.ReLU(),
                nn.LayerNorm(output_dim * 2),
                nn.Linear(output_dim * 2, output_dim)
            )

        #эмбеддинги отношений
        self.rel_emb = nn.Embedding(num_relations, output_dim)

    def forward(self, h_local, h_type, r_idx, t_local, t_type):
        batch_size = h_local.size(0)
        device = h_local.device

        #заготовки для итоговых тензоров
        h_proj = torch.zeros(batch_size, self.output_dim, device=device)
        t_proj = torch.zeros(batch_size, self.output_dim, device=device)

        #проход по всем типам данных
        for type_id in range(self.num_types):
            t_str = str(type_id)

            #выбирам элементы текущего типа
            h_mask = (h_type == type_id)
            t_mask = (t_type == type_id)

            #проекция голов
            if h_mask.any():
                raw_h = self.raw_stores[t_str](h_local[h_mask]).float()
                h_proj[h_mask] = self.projections[t_str](raw_h)

            #проекция хвостов
            if t_mask.any():
                raw_t = self.raw_stores[t_str](t_local[t_mask]).float()
                t_proj[t_mask] = self.projections[t_str](raw_t)

        #L2 нормализация
        h_proj = F.normalize(h_proj, p=2, dim=1)
        t_proj = F.normalize(t_proj, p=2, dim=1)

        #берем нужные эмбеддинги отношений
        r = self.rel_emb(r_idx)

        #считаем скор
        score = torch.sum(h_proj * r * t_proj, dim=1)
        return score


#функция для подготовки к оценке
def prepare_evaluation_mdm(model, all_h_local, all_h_type, all_r, all_t_local, all_t_type, device):
    model.eval()

    #проецируем все узлы графа в общее пространство
    all_embs = []
    entity2eval_id = {} #(local_id, type_id) -> индекс в матрице оценки
    eval_id = 0

    with torch.no_grad():
        #проход по всем типам данных
        for type_id in range(model.num_types):
            type_str = str(type_id)
            num_nodes = model.raw_stores[type_str].weight.size(0)

            #проецируем все узлы этого типа
            locs = torch.arange(num_nodes, device=device)
            raw = model.raw_stores[type_str](locs).float()
            proj = model.projections[type_str](raw)
            proj = F.normalize(proj, p=2, dim=1)

            all_embs.append(proj)

            #заполняем словарь какая строка новой матрицы какому узлу принадлежит
            for local_id in range(num_nodes):
                entity2eval_id[(local_id, type_id)] = eval_id
                eval_id += 1

    #матрица всех узлов графа [N_total, D]
    E_all = torch.cat(all_embs, dim=0)

    #ссловарь для фильтрации (h_eval_id, r) -> set(t_eval_ids)
    filter_dict = defaultdict(set)

    #проходимся по всем ребрам графа чтобы собрать словарь
    for i in range(len(all_h_local)):
        h_key = (all_h_local[i].item(), all_h_type[i].item())
        t_key = (all_t_local[i].item(), all_t_type[i].item())
        r = all_r[i].item()

        h_eval_id = entity2eval_id[h_key]
        t_eval_id = entity2eval_id[t_key]

        filter_dict[(h_eval_id, r)].add(t_eval_id)

    return E_all, entity2eval_id, filter_dict


#функция оценки
def evaluate_filtered_mdm(model, val_dataloader, E_all, entity_to_eval_id, filter_dict, device, k_list=[1, 5, 10, 50, 100]):
    model.eval()

    mrr = 0
    hits = {k: 0 for k in k_list}
    total_samples = 0

    with torch.no_grad():
        for i, batch in tqdm(enumerate(val_dataloader)):

            h_loc, h_typ, r_idx, t_loc, t_typ = [b.to(device) for b in batch]
            batch_size = h_loc.size(0)
            total_samples += batch_size

            #берем векторы голов и отношений
            h_eval_ids = [entity_to_eval_id[(local.item(), typ.item())] for local, typ in zip(h_loc, h_typ)]
            h_proj = E_all[torch.tensor(h_eval_ids, device=device)]
            r_emb = model.rel_emb(r_idx)

            #считаем скоры для всех связей разом
            query_emb = h_proj * r_emb
            all_scores = torch.matmul(query_emb, E_all.T) # [batch_size, N_total]

            #фильтрация
            mask_b = []
            mask_idx = []
            target_scores = torch.zeros(batch_size, device=device)

            #собираме индексы для маски фильтрации
            for i in range(batch_size):
                h_key = (h_loc[i].item(), h_typ[i].item())
                t_key = (t_loc[i].item(), t_typ[i].item())
                r = r_idx[i].item()

                h_eval_id = entity_to_eval_id[h_key]
                target_t_eval_id = entity_to_eval_id[t_key]

                #сохраняем целевой скор
                target_scores[i] = all_scores[i, target_t_eval_id]

                #проверяем наличие свази в графе
                true_tails = filter_dict[(h_eval_id, r)]
                for true_t in true_tails:
                    if true_t != target_t_eval_id:
                        mask_b.append(i)
                        mask_idx.append(true_t)

            #зануляем существующие связи
            if mask_b:
                all_scores[mask_b, mask_idx] = -1e9

            #сравниваем всю матрицу скоров с вектором целевых скоров
            ranks = (all_scores > target_scores.unsqueeze(1)).sum(dim=1) + 1

            #собираем метрики
            mrr += (1.0 / ranks).sum().item()
            for k in k_list:
                hits[k] += (ranks <= k).sum().item()

            del all_scores, query_emb


        mrr = mrr / total_samples
        hits = {k: v/total_samples for k, v in hits.items()}

        return mrr, hits

In [ ]:
#читаем предобученные эмбеддинги
with open("../data/dicts/dicts_for_mdm/dict_ESM_650M.pkl", 'rb') as f:
    emb_prot = pickle.load(f)
with open("../data/dicts/dicts_for_mdm/dict_chemberta_77m.pkl", 'rb') as f:
    emb_sm = pickle.load(f)
with open("../data/dicts/dicts_for_mdm/rna_berta.pkl", 'rb') as f:
    emb_rna = pickle.load(f)
with open("../data/dicts/dicts_for_mdm/dna_full.pkl", 'rb') as f:
    emb_dna = pickle.load(f)

#объединяем в один словарь
rawid2enb = emb_prot | emb_sm | emb_rna | emb_dna

In [ ]:
node_types = ['AA', 'DNA', 'RNA', 'SmallMolecule']
type2id = {name: i for i, name in enumerate(node_types)}

rawid_to_local = {}
rawid_to_type = {}
raw_tensors = {}


for t_name in node_types:
    #читаем колонку с ID чтобы узнать тип узлов
    df_nodes = pd.read_csv(f'../data/nodes/nodes_for_mdm/{t_name}.csv', usecols=['id_entity'])

    embeddings_list = []

    for local_idx, raw_id in enumerate(df_nodes['id_entity']):
        #заполняем словари для DataLoader
        rawid_to_local[raw_id] = local_idx
        rawid_to_type[raw_id] = type2id[t_name]

        #берем готовый эмбеддинг из исходного словаря
        emb = rawid2enb[raw_id]
        embeddings_list.append(emb)

    #склеиваем список тензоров в одну матрицу для этого типа [N_nodes_of_this_type, embedding_dim]
    raw_tensors[str(type2id[t_name])] = torch.stack(embeddings_list)

df_edges = pd.read_csv('../data/edges/clean_edges_without_NaNm.csv')

#превращаем предикаты в числа
pred_keys, pred_values = pd.factorize(df_edges['predicate'])
df_edges['predicate'] = pred_keys

#мапим сырые ID сразу в локальные индексы и типы
h_local = df_edges['id_entity_1'].map(rawid_to_local).to_numpy()
h_type  = df_edges['id_entity_1'].map(rawid_to_type).to_numpy()
t_local = df_edges['id_entity_2'].map(rawid_to_local).to_numpy()
t_type  = df_edges['id_entity_2'].map(rawid_to_type).to_numpy()
rel_idx = df_edges['predicate'].to_numpy()

#тензоры для DataLoader
h_local_tensor = torch.tensor(h_local, dtype=torch.long)
h_type_tensor  = torch.tensor(h_type, dtype=torch.long)
t_local_tensor = torch.tensor(t_local, dtype=torch.long)
t_type_tensor  = torch.tensor(t_type, dtype=torch.long)
rel_tensor     = torch.tensor(rel_idx, dtype=torch.long)

torch.manual_seed(100)
torch.cuda.manual_seed(100)

#создаем сеты для обучения и оценки
dataset = GraphDataset(h_local_tensor, h_type_tensor, rel_tensor, t_local_tensor, t_type_tensor)
train_set, val_set, test_set = random_split(dataset, [0.8, 0.1, 0.1])
train_loader = DataLoader(dataset=train_set, batch_size=4096, shuffle=True)
test_loader = DataLoader(dataset=test_set, batch_size=128, shuffle=True)

In [ ]:
#гиперпараметры
EMB_DIM = 192
LR = 2e-3
MARGIN = 1
WEIGHT = 1e-4
EPOCHS = 5


model = MultiModalDistMult(raw_tensors=raw_tensors, output_dim=EMB_DIM, num_relations=len(pred_values)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay = WEIGHT)
loss_fn = nn.MarginRankingLoss(margin=MARGIN)

model.train()

for epoch in tqdm(range(EPOCHS)):
    for batch in train_loader:
        optimizer.zero_grad()

        #позитивные примеры
        h_loc, h_typ, r_id, t_loc, t_typ = [b.to(device) for b in batch]
        batch_size = h_loc.size(0)

        #считаем скор для настоящих триплетов
        pos_scores = model(h_loc, h_typ, r_id, t_loc, t_typ)

        #генерируем негативные триплеты внутри батча
        perm = torch.randperm(batch_size, device=device)
        neg_t_loc = t_loc[perm]
        neg_t_typ = t_typ[perm]

        #считаем скор для негативным триплетам
        neg_scores = model(h_loc, h_typ, r_id, neg_t_loc, neg_t_typ)

        #считаем ошибку
        target = torch.ones_like(pos_scores)
        loss = loss_fn(pos_scores, neg_scores, target)

        loss.backward()
        optimizer.step()

In [ ]:
#подготовка к оценке
E_all, entity_to_eval_id, filter_dict = prepare_evaluation_mdm(
model, h_local_tensor, h_type_tensor, rel_tensor,
t_local_tensor, t_type_tensor, device)

#оценка
evaluate_filtered_mdm(model, test_loader, E_all, entity_to_eval_id, filter_dict, device, k_list=[1, 5, 10, 50, 100])